# 02 In-Context Learning (ICL): Exemplar Selection & Activation Steering

Implementing dynamic k-NN exemplar selection, LangChain ChatPromptTemplates, PyTorch hidden activation steering ($v_{\text{task}}$), and multi-provider model execution.

## Setup: Repository-Level Dotenv Configuration

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from repo-level .env file
load_dotenv()

print("Environment Setup Completed.")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))

## Technique 1: LangChain Semantic Similarity Exemplar Selection

In [ ]:
import numpy as np

class KNNExemplarSelector:
    def __init__(self, dataset: list[dict]):
        self.dataset = dataset

    def select_exemplars(self, query_embedding: np.ndarray, k: int = 2) -> list[dict]:
        scores = []
        for example in self.dataset:
            sim = np.dot(query_embedding, example["embedding"]) / (
                np.linalg.norm(query_embedding) * np.linalg.norm(example["embedding"])
            )
            scores.append((sim, example))
        scores.sort(key=lambda x: x[0], reverse=True)
        return [item[1] for item in scores[:k]]

dataset = [
    {"input": "Stock soared 15% after earnings.", "output": "Positive", "embedding": np.array([0.9, 0.1, 0.2])},
    {"input": "Layoffs impacted 500 employees.", "output": "Negative", "embedding": np.array([0.1, 0.8, 0.7])},
    {"input": "Revenue grew steadily in Q3.", "output": "Positive", "embedding": np.array([0.8, 0.2, 0.3])}
]

selector = KNNExemplarSelector(dataset)
top_exemplars = selector.select_exemplars(np.array([0.85, 0.15, 0.25]), k=2)
print("Top Selected Exemplars for ICL:")
for ex in top_exemplars:
    print(f"  Input: '{ex['input']}' -> Output: '{ex['output']}'")

## Technique 2: LangChain System & Few-Shot ChatPromptTemplates

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert sentiment classification model."),
    ("user", "Text: {text}"),
    ("assistant", "{label}")
])

rendered = system_prompt.format(text="Product arrived broken.", label="Negative")
print("LangChain Rendered Exemplar Prompt:\n", rendered)

## Technique 3: PyTorch Activation Steering Vector Extraction

In [ ]:
import torch

torch.manual_seed(42)
d_model = 64
num_samples = 15

h_base = torch.randn(num_samples, d_model) + 0.5
v_task_true = torch.tensor([2.0 if i % 2 == 0 else 1.5 for i in range(d_model)])
h_prompt = h_base + v_task_true + (torch.randn(num_samples, d_model) * 0.1)

v_task_estimated = (h_prompt - h_base).mean(dim=0)
h_steered = h_base + v_task_estimated

print(f"Extracted Steering Vector Norm: {v_task_estimated.norm().item():.3f}")

## Part 4: Visualizing Activation Displacement

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('dark_background')

plt.figure(figsize=(8, 4.5))
plt.scatter(h_base[:, 0].numpy(), h_base[:, 1].numpy(), color='#ff6b6b', label='Base Query Activations (h_base)', s=60)
plt.scatter(h_prompt[:, 0].numpy(), h_prompt[:, 1].numpy(), color='#51cf66', label='Few-Shot Activations (h_prompt)', s=60)
plt.scatter(h_steered[:, 0].numpy(), h_steered[:, 1].numpy(), color='#4dabf7', label='Steered Activations (h_steered)', marker='x', s=80)

plt.title('In-Context Learning (ICL): Activation Steering Vector Shift')
plt.xlabel('Hidden Dim 0'); plt.ylabel('Hidden Dim 1')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()